In [92]:
# Installing all required libraries for the Controlled Function-Calling Agent
!pip install wikipedia langchain langchain-openai openai

In [93]:
# Loading the OpenRouter API key securely using getpass.
# This ensures the key does NOT appear in logs or notebook output.
import os
from getpass import getpass

openrouter_key = getpass("Enter your OpenRouter API key (hidden): ")

# Setting environment variables for OpenAI-compatible OpenRouter APIs.
os.environ["OPENAI_API_KEY"] = openrouter_key
os.environ["OPENAI_BASE_URL"] = "https://openrouter.ai/api/v1"


Enter your OpenRouter API key (hidden): ··········


In [94]:
# Defining the external tools used by the agent.
# Tool 1: Wikipedia search – fetches a two-sentence summary for knowledge queries.
# Tool 2: Calculator – evaluates basic arithmetic expressions safely.

from langchain.tools import tool
import wikipedia

@tool
def search_wikipedia(query: str) -> str:
    """Search Wikipedia and return a two-sentence conceptual summary."""
    try:
        summary = wikipedia.summary(query, sentences=2)
        return summary
    except Exception as e:
        return f"Wikipedia error: {str(e)}"

@tool
def calculate_expression(expr: str) -> str:
    """Evaluate a simple arithmetic expression."""
    try:
        return str(eval(expr))
    except:
        return "Invalid arithmetic expression."


In [95]:
# Defining the routing logic using a language model (LLM).
# The model inspects user input and outputs ONE of the following:
# - "tool:wikipedia:<query>"
# - "tool:calculator:<expr>"
# - "answer:<direct_text>"
#
# This forms the core decision-making mechanism of the controlled agent.

from langchain_openai import ChatOpenAI
from langchain.prompts import ChatPromptTemplate

router_prompt = ChatPromptTemplate.from_messages([
    ("system",
    """
You are a routing model.

Choose ONLY one of the following formats for each user message:

1) tool:wikipedia:<query>
2) tool:calculator:<expr>
3) answer:<text>

Routing Rules:
- If the user asks 'what is', 'who is', or 'tell me about', use Wikipedia.
- If the user enters math expressions (+, -, *, /), use Calculator.
- Otherwise, provide a direct text answer.

Output only one line without explanation.
    """),
    ("user", "{input}")
])

# Initializing the LLM (OpenRouter-compatible) with deterministic behavior.
llm = ChatOpenAI(
    model="google/gemma-2-9b-it",
    temperature=0
)

# The router consists of a prompt followed by LLM inference.
router = router_prompt | llm


In [96]:
# Agent Function:
# Routes user input based on the LLM's decision.
# Cleans the router output, identifies the correct tool,
# executes it, and returns the final response.
def agent(user_input):

    routing_output = router.invoke({"input": user_input}).content

    # Clean output for consistent parsing
    routing_output = routing_output.strip()

    print(" Router Output:", routing_output)

    if routing_output.startswith("tool:wikipedia:"):
        query = routing_output.replace("tool:wikipedia:", "", 1).strip()
        return search_wikipedia.run(query)

    elif routing_output.startswith("tool:calculator:"):
        expr = routing_output.replace("tool:calculator:", "", 1).strip()
        return calculate_expression.run(expr)

    elif routing_output.startswith("answer:"):
        return routing_output.replace("answer:", "", 1).strip()

    return "Invalid router output."


In [97]:
# A simple helper to maintain natural conversation flow.
# Stores conversation history (not required but useful for demonstration).

conversation_history = []

def chat(msg):
    print("\n User:", msg)
    response = agent(msg)
    print(" Assistant:", response)
    conversation_history.append({"user": msg, "assistant": response})
    return response


In [98]:
# Demonstration of five interactions, showing how the agent switches
# dynamically between Wikipedia, Calculator, and direct answers.

chat("What is machine learning?")
chat("22 * 4 + 3")
chat("How are you?")
chat("Who is Virat Kohli?")
chat("100 / 25")



 User: What is machine learning?
 Router Output: tool:wikipedia:machine learning
 Assistant: Wikipedia error: Page id "machine ;earning" does not match any pages. Try another id!

 User: 22 * 4 + 3
 Router Output: tool:calculator:22*4+3
 Assistant: 91

 User: How are you?
 Router Output: 
 Assistant: Invalid router output.

 User: Who is Virat Kohli?
 Router Output: tool:wikipedia:Virat Kohli
 Assistant: Virat Kohli (Hindi pronunciation: [ʋɪˈɾaːʈᵊ ˈkoːɦᵊliː] , born 5 November 1988) is an Indian international cricketer and the former captain of the Indian national cricket team. He is a right-handed batsman and an occasional medium-pace bowler.

 User: 100 / 25
 Router Output: 
 Assistant: Invalid router output.


'Invalid router output.'

In [99]:
# User-defined query tester
while True:
    user_msg = input("\nEnter your question (or type 'exit' to stop): ")

    if user_msg.lower() == "exit":
        print("Interaction stopped.")
        break

    print("\n🧑 User:", user_msg)
    response = agent(user_msg)
    print("🤖 Assistant:", response)



Enter your question (or type 'exit' to stop): how are you

🧑 User: how are you
 Router Output: answer:I am a large language model.
🤖 Assistant: I am a large language model.

Enter your question (or type 'exit' to stop): "Who is Alan Turing's mentor?"

🧑 User: "Who is Alan Turing's mentor?"
 Router Output: tool:wikipedia:Alan Turing's mentor
🤖 Assistant: Wikipedia error: Page id "alan turing men of" does not match any pages. Try another id!

Enter your question (or type 'exit' to stop): "((45 * 3) - 28) / 2"

🧑 User: "((45 * 3) - 28) / 2"
 Router Output: tool:calculator:((45 * 3) - 28) / 2
🤖 Assistant: 53.5

Enter your question (or type 'exit' to stop): exit
Interaction stopped.
